# 05. Final Load Prep — Master Export (Tableau Ready)
**Project:** Logistics & Delivery Delays Analysis (Olist)
**Focus:** Two complementary CSV exports for Tableau analysis


In [11]:
import pandas as pd
import numpy as np
import os
import logging
import warnings
warnings.filterwarnings('ignore')

PATH_CLEANED = "../data/processed/olist_cleaned_data.csv"
PATH_TABLEAU = "../data/processed/tableau_ready/"
os.makedirs(PATH_TABLEAU, exist_ok=True)

logger = logging.getLogger('Tableau Export')
logging.basicConfig(level=logging.INFO, format='%(message)s')

df = pd.read_csv(PATH_CLEANED)
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])
df['order_year'] = df['order_purchase_timestamp'].dt.year
df['order_quarter'] = df['order_purchase_timestamp'].dt.to_period('Q').dt.strftime('Q%q-%Y')
df['is_bad_review'] = df['review_score'] == 1
print(f"✓ Base dataset loaded. Shape: {df.shape}")

def clean_colnames(dataframe):
    df_out = dataframe.copy()
    df_out.columns = [col.replace('_', ' ').title() for col in df_out.columns]
    return df_out


✓ Base dataset loaded. Shape: (109657, 51)


## Master Export Layer

### Overview
Two complementary datasets for Tableau analysis:
- **File 1:** Row-level data (30K sample) — scatter plots, trend filters, drill-down
- **File 2:** Aggregated metrics — risk analysis, route performance, seller health

**Join Key:** Seller State + Order Year


### File 1: Orders Master (30K Row-level Sample)
**Output:** `olist_orders_master.csv`


In [12]:
orders_master = df[[
    'order_id', 'order_year', 'order_quarter',
    'order_purchase_timestamp', 'delivery_delay',
    'actual_delivery_days', 'estimated_delivery_days',
    'review_score', 'is_late', 'is_bad_review',
    'seller_state', 'seller_city', 'seller_id',
    'customer_state', 'shipping_route', 'haversine_distance_km'
]].copy()

orders_master['delivery_status'] = orders_master['is_late'].map(
    {True: 'Late', False: 'On Time'})
orders_master['review_tier'] = orders_master['is_bad_review'].map(
    {True: '1-2 Star (Bad)', False: '3-5 Star (Good)'})

orders_master = orders_master.drop(columns=['is_late', 'is_bad_review'])
orders_master = orders_master.sample(n=min(30000, len(orders_master)), random_state=42)

output_file_1 = os.path.join(PATH_TABLEAU, 'olist_orders_master.csv')
clean_colnames(orders_master).to_csv(output_file_1, index=False)

print(f'File 1 exported: olist_orders_master.csv | {orders_master.shape}')


File 1 exported: olist_orders_master.csv | (30000, 16)


### File 2: Aggregated Summary (State + Seller + Quarter Level)
**Output:** `olist_aggregated_summary.csv`


In [13]:
agg_summary = df.groupby(
    ['seller_state', 'customer_state', 'order_year', 'order_quarter', 'seller_id']
).agg(
    total_orders      = ('order_id',        'count'),
    late_orders       = ('is_late',          'sum'),
    avg_delay_days    = ('delivery_delay',   'mean'),
    avg_review_score  = ('review_score',     'mean'),
    bad_reviews       = ('is_bad_review',    'sum'),
    avg_distance_km   = ('haversine_distance_km', 'mean'),
).reset_index()

agg_summary['late_rate_pct']    = (agg_summary['late_orders']  / agg_summary['total_orders'] * 100).round(2)
agg_summary['bad_review_rate']  = (agg_summary['bad_reviews']  / agg_summary['total_orders'] * 100).round(2)
agg_summary['avg_delay_days']   = agg_summary['avg_delay_days'].round(2)
agg_summary['avg_review_score'] = agg_summary['avg_review_score'].round(3)
agg_summary['high_risk_seller'] = (agg_summary['late_rate_pct'] > 13).map(
    {True: 'High Risk', False: 'Normal'})
agg_summary['shipping_route']   = agg_summary['seller_state'] + ' → ' + agg_summary['customer_state']

output_file_2 = os.path.join(PATH_TABLEAU, 'olist_aggregated_summary.csv')
clean_colnames(agg_summary).to_csv(output_file_2, index=False)

print(f'File 2 exported: olist_aggregated_summary.csv | {agg_summary.shape}')
print('\nDone. Load BOTH files into Tableau and relate on Seller State + Order Year.')


File 2 exported: olist_aggregated_summary.csv | (32860, 15)

Done. Load BOTH files into Tableau and relate on Seller State + Order Year.
